# NYC Taxi Fare Prediction: Phase 3: Data Cleaning
### Automatidata x New York City Taxi & Limousine Commission
---
**Goal:** Enforce the data contracts defined in Phase 1 to produce a clean, leakage-free dataset ready for feature engineering in Phase 4.

**Operations (in order):**
1. Cast `extra` and `total_amount` to `float64`
2. Drop 518 near-duplicate trips
3. Drop rows violating Phase 1 range contracts
4. Drop leakage and near-constant columns identified in Phase 2
5. Validate the clean dataset and save to `data/taxi_clean.parquet`

**Input:**  `data/2017_Yellow_Taxi_Trip_Data.csv`: 1,000,000 rows × 17 columns  
**Output:** `data/taxi_clean.parquet`: ~991,740 rows × 12 columns

In [ ]:
# ------------------------------------------------------------
# Step 3.0: Notebook Initialization
# ------------------------------------------------------------

import sys
sys.path.append("..")

from src.config import *

# Load & register data
df_raw = load_raw_data(url="URL")    # Paste the URL if the downloaded file is not present on disk
register_duckdb_table(df_raw, table_name="trips")

print("Config loaded successfully.")

print("\n" + "=" * 70)
print("Baseline Dataset")
print("=" * 70)

print(f"   Rows    : {df_raw.shape[0]:,}")
print(f"   Columns : {df_raw.shape[1]}")
print(f"\n   Columns : {df_raw.columns.tolist()}")

## Step 3.1: Dtype Casting
Cast `extra` and `total_amount` from `object` to `float64` as identified in Phase 1. Non-numeric values that cannot be coerced are set to `NaN` and counted — any new nulls introduced by the cast are flagged before we proceed to dropping rows.

In [ ]:
# ------------------------------------------------------------
# Step 3.1: Dtype Casting
# ------------------------------------------------------------

def cast_numeric_columns(
    df: pd.DataFrame,
    cols: list[str]
) -> pd.DataFrame:
    """
    Cast specified columns to float64, coercing unparseable values to NaN.

    For each column, logs the dtype before and after casting and reports
    how many new NaN values were introduced by the coercion. Raises a
    warning if any NaNs are introduced, as these represent values that
    could not be parsed as numeric and will require attention.

    Parameters
    ----------
    df   : pd.DataFrame
        The DataFrame to modify.
    cols : list[str]
        Column names to cast to float64.

    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with specified columns cast to float64.
    """
    df = df.copy()

    for col in cols:
        if col not in df.columns:
            logger.warning(f"Column '{col}' not found: skipping cast.")
            continue

        dtype_before = df[col].dtype
        nulls_before = df[col].isna().sum()

        df[col] = pd.to_numeric(df[col], errors="coerce")

        nulls_after = df[col].isna().sum()
        new_nulls = nulls_after - nulls_before

        logger.info(
            f"Cast '{col}': {dtype_before} -> float64 "
            f"| new NaNs introduced: {new_nulls:,}"
        )

        if new_nulls > 0:
            logger.warning(
                f"  ⚠️  '{col}' has {new_nulls:,} unparseable values, "
                f"these rows will be dropped in Step 3.4."
            )

    return df

# Run
COLS_TO_CAST: list[str] = ["extra", "total_amount"]

df_clean = cast_numeric_columns(df_raw, COLS_TO_CAST)

# Verify dtypes
print("\n" + "=" * 70)
print("Dtype Verification")
print("=" * 70)

print(f"{'Column':<25} {'Before':<15} {'After'}")
print("-" * 50)
for col in COLS_TO_CAST:
    print(
        f"{col:<25} "
        f"{str(df_raw[col].dtype):<15}"
        f"{str(df_clean[col].dtype)}"
    )

print(f"\nRows before: {df_raw.shape[0]:,}")
print(f"Rows after: {df_clean.shape[0]:,}    (no rows dropped in this step)")


## Step 3.1: Dtype Casting ✅

Both object-typed columns successfully cast to `float64`.

| Column | Before | After | New NaNs |
|---|---|---|---|
| `extra` | `object` | `float64` | 1 |
| `total_amount` | `object` | `float64` | 1 |

> **Note:** 1 unparseable value was introduced in each column during coercion.
> These 2 rows (possibly the same row) will be caught and dropped automatically
> in Step 3.4 alongside the other range violation drops — no special handling needed.
> 

## Step 3.2: Drop Near-Duplicates
Remove the 518 near-duplicate trips identified in Phase 1, rows sharing the same pickup datetime, dropoff datetime, and fare amount. The first occurrence of each duplicate group is retained.

In [ ]:
# ------------------------------------------------------------
# Step 3.2: Drop Near-Duplicates
# ------------------------------------------------------------

def drop_near_duplicates(
    df: pd.DataFrame,
    subset: list[str] = None
) -> tuple[pd.DataFrame, int]:
    """
    Remove near-duplicate rows based on a subset of key columns.

    The first occurrence of each duplicate group is retained and all
    subsequent duplicates are dropped. The subset defaults to the three
    columns used in the Phase 1 audit: pickup datetime, dropoff datetime
    and fare amount.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to deduplicate.
    subset: list[str], optional
        Columns to consider when identifying near-duplicates.
        Defaults to ``["tpep_pickup_datetime", "tpep_dropoff_datetime",
        "fare_amount"]``.

    Returns
    -------
    tuple[pd.DataFrame, int]
        Cleaned DataFrame and the number of rows dropped.
    """
    subset = subset or [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "fare_amount"
    ]

    rows_before = len(df)
    df_deduped = df.drop_duplicates(subset=subset, keep="first").copy()
    rows_after = len(df_deduped)
    n_dropped = rows_before - rows_after

    logger.info(f"Near-duplicates dropped: {n_dropped:,}")
    logger.info(f"Rows remaining: {rows_after:,}")

    return df_deduped, n_dropped

# Run deduplication
df_clean, n_dupes_dropped = drop_near_duplicates(df_clean)

print("\n" + "=" * 70)
print("Deduplication Summary")
print("=" * 70)
print(f"  Rows before: {df_raw.shape[0]:,}")
print(f"  Dropped: {n_dupes_dropped:,}")
print(f"  Rows after: {df_clean.shape[0]:,}")

# Verify no near-duplicates remain
remaining = df_clean.duplicated(
    subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", "fare_amount"],
    keep=False
).sum()

print(f"\n  Remaining near-duplicates: {remaining:,}  "
     f"{'✅' if remaining == 0 else '⚠️  Unexpected: investigate'}")


## Step 3.2: Drop Near-Duplicates ✅

Near-duplicate rows removed using the key subset:
`tpep_pickup_datetime`, `tpep_dropoff_datetime`, `fare_amount`.

| Metric | Value |
|---|---|
| Rows before | 1,000,000 |
| Duplicate pairs found (Phase 1) | 518 rows across ~259 pairs |
| Rows dropped (one per pair) | 259 |
| Rows after | 999,741 |
| Remaining near-duplicates | 0 ✅ |

> **Note:** `keep="first"` retains the first occurrence of each duplicate
> group and drops all subsequent matches. 518 flagged rows across ~259
> pairs yields 259 removals — one duplicate eliminated per pair.

## Step 3.3: Drop Invalid Rows
Enforce all Phase 1 range contracts in a single pass. Rows violating any contract are dropped together to avoid double-counting rows that breach multiple rules simultaneously. Leakage and near-constant columns identified in Phase 2 are also dropped here.

In [ ]:
# ------------------------------------------------------------
# Step 3.3: Drop Invalid Rows & Leakage Columns
# ------------------------------------------------------------

def drop_invalid_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Remove rows that violate the Phase 1 data contracts in a single pass.

    All conditions are evaluated simultanuously so rows breaching multiple
    rules are counted only once. A breaddown of how many rows each individual
    rule would have removed in returned alongside the cleaned DataFrame for transparency.

    Parameters
    ----------
    df: pd.DataFrame
        The deduplicated DataFrame from Step 3.3.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        - Cleaned DataFrame with all invalid rows removed.
        - Breakdown table showing per-rule row counts.
    """
    # Define individual rule masks (True = invalid)
    rules: dict[str, pd.Series] = {
        "trip_distance < 0.01": df["trip_distance"] < 0.01,
        "fare_amount <= 0": df["fare_amount"] <= 0,
        "passenger_count < 1": df["passenger_count"] < 1,
        "ratecodeid not in 1–6": ~df["ratecodeid"].isin([1, 2, 3, 4, 5, 6]),
        "tip_amount < 0": df["tip_amount"] < 0,
        "tolls_amount < 0": df["tolls_amount"] < 0,
        "mta_tax < 0 or > 0.5": (df["mta_tax"] < 0) | (df["mta_tax"] > 0.5),
        "extra < 0 or > 10": (df["extra"] < 0) | (df["extra"] > 10),
        "total_amount <= 0": df["total_amount"] <= 0,
        "extra or total_amount NaN": df["extra"].isna() | df["total_amount"].isna()
    }

    # Per-rule breakdown (for reporting)
    breakdown = pd.DataFrame([
        {"rule": rule, "rows_affected": int(mask.sum())}
        for rule, mask in rules.items()
    ])

    # Combined invalid mask (union of all rules)
    combined_mask = pd.concat(rules.values(), axis=1).any(axis=1)

    rows_before = len(df)
    df_clean = df[~combined_mask].copy()
    rows_after = len(df_clean)
    n_dropped = rows_before - rows_after

    logger.info(f"Invalid rows dropped: {n_dropped:,}")
    logger.info(f"Rows remaining: {rows_after:,}")

    return df_clean, breakdown

def drop_leakage_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Drop leakage and near-constant columns identified in Phase 2.-
    
    Columns removed:
      - ``total_amount``          : computed from fare_amount — hard leakage
      - ``tip_amount``            : recorded post-ride — temporal leakage
      - ``mta_tax``               : near-constant ($0.50) — no signal
      - ``improvement_surcharge`` : near-constant ($0.30) — no signal
      - ``store_and_fwd_flag``    : 99.73% constant — no signal

    Parameters
    ----------
    df : pd.DataFrame
        The cleaned DataFrame with invalid rows already removed.

    Returns
    -------
    tuple[pd.DataFrame, list[str]]
        DataFrame with leakage columns removed, and the list of
        columns that were dropped.
    """
    cols_to_drop = [
        "total_amount",
        "tip_amount",
        "mta_tax",
        "improvement_surcharge",
        "store_and_fwd_flag",
    ]

    # Only drop columns that actually exist
    cols_present = [c for c in cols_to_drop if c in df.columns]
    df_dropped = df.drop(columns=cols_present)

    for col in cols_present:
        logger.info(f"Dropped column: '{col}'")

    return df_dropped, cols_present

# Run invalid row removal
df_clean, rule_breakdown = drop_invalid_rows(df_clean)

print("\n" + "=" * 70)
print("Per-Rule Breakdown")
print("=" * 70)
print(rule_breakdown.to_string(index=False))
print(f"\n  Note: rows may violate multiple rules: total dropped")
print(f"  reflects the union of all masks, not the sum above.")

print("\n" + "=" * 70)
print("Row Count After Invalid Row Removal")
print("=" * 70)
print(f"  Rows after deduplication: 999,741")
print(f"  Rows after invalid drop: {df_clean.shape[0]:,}")
print(f"  Dropped in this step: {999_741 - df_clean.shape[0]:,}")

# Run leakage column removal
df_clean, dropped_cols = drop_leakage_columns(df_clean)

print("\n" + "=" * 70)
print("Columns After Leakage Drop")
print("=" * 70)
print(f" Columns dropped: {dropped_cols}")
print(f"  Columns before: 17")
print(f"  Columns after: {df_clean.shape[1]}")
print(f"\n  Remaining columns:")
for col in df_clean.columns.tolist():
    print(f"    - {col}")


## Step 3.3: Drop Invalid Rows & Leakage Columns ✅

### Per-Rule Breakdown

| Rule | Rows Affected |
|---|---|
| `trip_distance < 0.01` | 7,049 |
| `fare_amount <= 0` | 893 |
| `total_amount <= 0` | 724 |
| `mta_tax < 0 or > 0.5` | 564 |
| `extra < 0 or > 10` | 263 |
| `passenger_count < 1` | 88 |
| `ratecodeid not in 1–6` | 15 |
| `tip_amount < 0` | 9 |
| `tolls_amount < 0` | 6 |
| `extra or total_amount NaN` | 1 |

> Rows may violate multiple rules simultaneously, the 7,743 dropped
> reflects the union of all masks, not the sum of the individual counts above.

### Row Count Summary

| Stage | Rows |
|---|---|
| Raw dataset | 1,000,000 |
| After near-duplicate drop (Step 3.3) | 999,741 |
| After invalid row drop (Step 3.4) | 991,998 |
| **Total dropped** | **8,002** |
| **Retained** | **99.20%** |

### Leakage & Near-Constant Columns Dropped

| Column | Reason |
|---|---|
| `total_amount` | Hard leakage — derived directly from `fare_amount` |
| `tip_amount` | Temporal leakage — recorded post-ride, not available at pickup |
| `mta_tax` | Near-constant (\\$0.50) — no predictive signal |
| `improvement_surcharge` | Near-constant (\\$0.30) — no predictive signal |
| `store_and_fwd_flag` | 99.73% constant — no predictive signal |

### Remaining Columns (12)

`vendorid`, `tpep_pickup_datetime`, `tpep_dropoff_datetime`,
`passenger_count`, `trip_distance`, `ratecodeid`, `pulocationid`,
`dolocationid`, `payment_type`, `fare_amount`, `extra`, `tolls_amount`

## Step 3.4: Post-Cleaning Validation
Re-run the Phase 1 audit checks against the cleaned dataset to confirm all contracts have been enforced. Expected result: zero violations across all columns, zero near-duplicates, and correct dtypes throughout.

In [ ]:
# ------------------------------------------------------------
# Step 3.4: Post-Cleaning Validation
# ------------------------------------------------------------

def validate_clean_dataset(df: pd.DataFrame) -> None:
    """
    Run a full suite of post-cleaning checks to confirm all Phase 1
    data contracts have been enforced on the cleaned DataFrame.

    Checks performed:
        1. Shape and dtype inspection
        2. Missing value count
        3. Near-duplicate count
        4. Range contract validation against cleaned columns

    Parameters
    ----------
    df : pd.DataFrame
        The cleaned DataFrame to validate.
    """
    print("\n" + "=" * 70)
    print("1. Shape & Dtypes")
    print("=" * 70)
    print(f"  Rows: {df.shape[0]:,}")
    print(f"  Columns: {df.shape[1]}")
    print()

    schema = pd.DataFrame({
        "dtype": df.dtypes,
        "non_null": df.notna().sum(),
        "null_count": df.isna().sum()
    })
    print(schema.to_string())

    print("\n" + "=" * 70)
    print("2. Missing Values")
    print("=" * 70)
    total_nulls = df.isna().sum().sum()
    if total_nulls == 0:
        print("  ✅ No missing values detected.")
    else:
        print(f"  ⚠️ {total_nulls:,} missing values remain:")
        print(df.isna().sum()[df.isna().sum() > 0].to_string())

    print("\n" + "=" * 70)
    print("3. Near-Duplicates")
    print("=" * 70)
    near_dup_count = df.duplicated(
        subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", "fare_amount"],
        keep=False
    ).sum()

    if near_dup_count == 0:
        print("  ✅ No near-duplicates detected.")
    else:
        print(f"  ⚠️ {near_dup_count:,} near-duplicates remain.")

    print("\n" + "=" * 70)
    print("4. Range Contract Validation")
    print("=" * 70)
    # Only validate columns still present after leakage drop
    contracts = {
        "passenger_count" : (1, 9, ">=1 and <=9"),
        "trip_distance" : (0.01, 200, ">= 0.01 miles"),
        "ratecodeid" : (1, 6, "in range 1–6"),
        "fare_amount" : (0.01, 1000, "> 0"),
        "extra" : (0, 10, "between 0 and 10"),
        "tolls_amount" : (0, 500, ">= 0"),
    }

    all_clear = True
    for col, (low, high, desc) in contracts.items():
        if col not in df.columns:
            continue

        series = pd.to_numeric(df[col], errors="coerce")
        n_violate = ((series < low) | (series > high)).sum()
        status = "✅" if n_violate == 0 else "⚠️ "
        if n_violate > 0:
            all_clear = False
        print(f"  {status} {col:<22} {desc:<25} violations: {n_violate:,}")

    if all_clear:
        print("\n  ✅ All range contracts satisfied.")

    print("\n" + "=" * 70)
    print("5. Fare Amount Distribution (post-clean)")
    print("=" * 70)
    fare = df["fare_amount"]
    print(f"  Min: ${fare.min():.2f}")
    print(f"  Median: ${fare.median():.2f}")
    print(f"  Mean: ${fare.mean():.2f}")
    print(f"  Max: ${fare.max():.2f}")
    print(f"  Negative fares: {(fare < 0).sum()}")

# Register clean df with DuckDB for any SQL checks
register_duckdb_table(df_clean, table_name="trips_clean")

# Run validation
validate_clean_dataset(df_clean)
    

## Step 3.4: Post-Cleaning Validation ✅

All post-cleaning checks passed. The dataset is fully contract-compliant.

### Shape & Dtypes

| Column | Dtype | Nulls |
|---|---|---|
| `vendorid` | `int64` | 0 |
| `tpep_pickup_datetime` | `datetime64[ns]` | 0 |
| `tpep_dropoff_datetime` | `datetime64[ns]` | 0 |
| `passenger_count` | `int64` | 0 |
| `trip_distance` | `float64` | 0 |
| `ratecodeid` | `int64` | 0 |
| `pulocationid` | `int64` | 0 |
| `dolocationid` | `int64` | 0 |
| `payment_type` | `int64` | 0 |
| `fare_amount` | `float64` | 0 |
| `extra` | `float64` | 0 |
| `tolls_amount` | `float64` | 0 |

### Validation Results

| Check | Result |
|---|---|
| Missing values | ✅ None |
| Near-duplicates | ✅ None |
| Range contracts (all 6) | ✅ All satisfied |
| Negative fares | ✅ None remaining |

### Fare Amount Distribution (post-clean)

| Metric | Value |
|---|---|
| Min | \\$0.01 |
| Median | \\$9.50 |
| Mean | \\$13.09 |
| Max | \\$621.50 |
| Negative fares | 0 |

> The mean dropped slightly from \\$13.15 (raw) to \\$13.09 (clean), which is consistent
> with removing negative fares and meter-error trips that were pulling the
> distribution leftward. The median is unchanged at $9.50, confirming the
> bulk of the distribution is unaffected by cleaning.

## Step 3.5: Save Clean Dataset
Persist the cleaned DataFrame to `data/taxi_clean.parquet`. Parquet is preferred over CSV for downstream phases because it preserves dtypes (no re-casting needed), is significantly faster to read, and compresses the dataset substantially on disk.

In [ ]:
# ------------------------------------------------------------
# Step 3.5: Save Clean Dataset
# ------------------------------------------------------------

def save_clean_parquet(
    df: pd.DataFrame,
    path: Path = CLEAN_DATA_PATH,
) -> None:
    """
    Save the cleaned DataFrame to a Parquet file.

    Uses the snappy compression codec for a good balance of compression ratio
    and read speed. Logs file size on disk after writing so the storage saving
    over the original CSV can be assessed.

    Parameters
    ----------
    df   : pd.DataFrame
        The cleaned DataFrame to persist.
    path : Path
        Destination file path (default ``CLEAN_DATA_PATH``).
    """
    df.to_parquet(path, index=False, compression="snappy")

    size_mb = path.stat().st_size / 1e6
    logger.info(f"Clean dataset saved -> {path}")
    logger.info(f"File size on disk: {size_mb:.1f} MB")

def verify_parquet_roundtrip(path: Path = CLEAN_DATA_PATH) -> pd.DataFrame:
    """
    Reload the saved Parquet file and verify its integrity.

    Confirms that the row count, column count, and dtypes are identical to
    the in-memory cleaned DataFrame, ensuring no data was lost or corrupted
    during the write.

    Parameters
    ----------
    path : Path
        Path to the saved Parquet file.

    Returns
    -------
    pd.DataFrame
        The reloaded DataFrame, ready for use in Phase 4.
    """
    df_loaded = pd.read_parquet(path)

    logger.info(f"Parquet reloaded: {df_loaded.shape[0]:,} rows "
               f"x {df_loaded.shape[1]} columns")

    return df_loaded

# # Save
save_clean_parquet(df_clean)

# Reload & Verify
df_reloaded = verify_parquet_roundtrip()

print("\n" + "=" * 70)
print("Parquet Roundtrip Verification")
print("=" * 70)
print(f"\n  In-memory shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
print(f"  Reloaded shape: {df_reloaded.shape[0]:,} rows x {df_reloaded.shape[1]} columns")

shape_match = df_clean.shape == df_reloaded.shape
dtype_match = (df_clean.dtypes == df_reloaded.dtypes).all()

print(f"\n  Shape match: {'✅' if shape_match else '⚠️ MISMATCH'}")
print(f"  Dtype match: {'✅' if dtype_match else '⚠️ MISMATCH'}")

print("\n" + "=" * 70)
print("Reloaded Dtypes")
print("=" * 70)
print(df_reloaded.dtypes.to_string())


## Step 3.5: Save Clean Dataset ✅

The cleaned DataFrame was saved to `data/taxi_clean.parquet` and
successfully reloaded with identical shape and dtypes.

### Parquet Roundtrip Verification

| Check | Result |
|---|---|
| Shape match | ✅ 991,998 rows × 12 columns |
| Dtype match | ✅ All 12 columns identical |

### Reloaded Dtypes

| Column | Dtype |
|---|---|
| `vendorid` | `int64` |
| `tpep_pickup_datetime` | `datetime64[ns]` |
| `tpep_dropoff_datetime` | `datetime64[ns]` |
| `passenger_count` | `int64` |
| `trip_distance` | `float64` |
| `ratecodeid` | `int64` |
| `pulocationid` | `int64` |
| `dolocationid` | `int64` |
| `payment_type` | `int64` |
| `fare_amount` | `float64` |
| `extra` | `float64` |
| `tolls_amount` | `float64` |

> Parquet preserves dtypes natively, no re-casting will be needed
> when Phase 4 loads this file. Snappy compression ensures fast
> read speeds for the 991K-row dataset.